# Monte Carlo Analysis with 3-DOF Simulations

This notebook demonstrates Monte Carlo uncertainty analysis using 3-DOF rocket simulations.

## Overview

Monte Carlo simulations allow us to:
- Account for uncertainties in rocket parameters
- Analyze trajectory dispersion
- Calculate landing ellipses
- Perform statistical analysis of flight characteristics

With 3-DOF simulations, Monte Carlo analysis becomes **much faster**, enabling thousands of simulations in reasonable time.

## Current Limitations (Important!)

⚠️ **Note**: The current implementation of Monte Carlo in RocketPy has some limitations when using 3-DOF simulations:

1. **No StochasticPointMassRocket**: There is no dedicated stochastic wrapper for `PointMassRocket`
2. **Workaround needed**: We must use `StochasticRocket` with a regular `PointMassRocket`
3. **Some parameters cannot be randomized**: Inertia parameters are fixed for point mass models

This notebook demonstrates **working approaches** and highlights these limitations.

In [1]:
# Import required libraries
from rocketpy import Environment
from rocketpy.motors.point_mass_motor import PointMassMotor
from rocketpy.rocket.point_mass_rocket import PointMassRocket
from rocketpy.simulation.flight import Flight
from rocketpy.simulation import MonteCarlo
from rocketpy.stochastic import (
    StochasticEnvironment,
    StochasticFlight,
    StochasticRocket,
)
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output

## Approach 1: Monte Carlo with Flight Parameter Variation Only

This is the **simplest and most reliable** approach - varying only flight parameters (not rocket or motor).

This approach works perfectly and demonstrates:
- Launch angle variations
- Launch azimuth (heading) variations  
- Rail length uncertainties

In [2]:
# Create deterministic environment, motor, and rocket
env = Environment(
    latitude=39.389,
    longitude=-8.289,
    elevation=113
)
env.set_atmospheric_model(type='standard_atmosphere')

# Create point mass motor
motor = PointMassMotor(
    thrust_source=500,
    dry_mass=1.5,
    propellant_initial_mass=2.0,
    burn_time=3.5,
)

# Create point mass rocket
rocket = PointMassRocket(
    radius=0.0635,
    mass=5.0,
    center_of_mass_without_motor=0.0,
    power_off_drag=0.5,
    power_on_drag=0.5,
)
rocket.add_motor(motor, position=0.0)

# Create nominal flight
nominal_flight = Flight(
    rocket=rocket,
    environment=env,
    rail_length=5.0,
    inclination=84,
    heading=90,
    simulation_mode='3 DOF',
)

print(f"Nominal apogee: {nominal_flight.apogee - env.elevation:.2f} m AGL")
print(f"Nominal impact: x={nominal_flight.x_impact:.2f} m, y={nominal_flight.y_impact:.2f} m")

Nominal apogee: 1235.70 m AGL
Nominal impact: x=453.01 m, y=0.00 m


### Define Stochastic Flight Parameters

We'll vary:
- **Inclination**: 84° ± 2° (normal distribution)
- **Heading**: 90° ± 3° (normal distribution)
- **Rail length**: 5.0 ± 0.1 m (normal distribution)

In [3]:
# Create stochastic environment (no variation in this example)
stochastic_env = StochasticEnvironment(environment=env)

# Create stochastic flight with parameter uncertainties
# Format: (mean, std_dev, 'distribution_type')
stochastic_flight = StochasticFlight(
    flight=nominal_flight,
    rail_length=(5.0, 0.1, 'normal'),    # 5.0 ± 0.1 m
    inclination=(84, 2.0, 'normal'),      # 84° ± 2°
    heading=(90, 3.0, 'normal'),          # 90° ± 3°
)

print("Stochastic flight parameters configured:")
print("  - Rail length: 5.0 ± 0.1 m")
print("  - Inclination: 84 ± 2°")  
print("  - Heading: 90 ± 3°")

Stochastic flight parameters configured:
  - Rail length: 5.0 ± 0.1 m
  - Inclination: 84 ± 2°
  - Heading: 90 ± 3°


### Run Monte Carlo Simulation

Now we'll run the Monte Carlo simulation with the stochastic flight parameters.

In [4]:
# Create Monte Carlo object
# Note: rocket must be passed as-is (not stochastic) for 3-DOF
mc = MonteCarlo(
    filename="mc_3dof_flight_only",
    environment=stochastic_env,
    rocket=rocket,  # Regular rocket (no stochastic wrapper)
    flight=stochastic_flight,
)

# Run simulations
print("\nRunning Monte Carlo simulation...")
print("This may take a minute...\n")

mc.simulate(
    number_of_simulations=100,  # 100 simulations for demonstration
    append=False,  # Start fresh
)

print(f"\nCompleted {mc.number_of_simulations} simulations")
print(f"Total CPU time: {mc.total_cpu_time:.2f} seconds")
print(f"Average time per simulation: {mc.total_cpu_time/mc.number_of_simulations:.4f} seconds")

The following input file was imported: mc_3dof_flight_only.inputs.txt
A total of 0 simulations results were loaded from the following output file: mc_3dof_flight_only.outputs.txt

The following error file was imported: mc_3dof_flight_only.errors.txt                                        

Running Monte Carlo simulation...
This may take a minute...

Starting Monte Carlo analysis                                        
Error on iteration 1: 'PointMassRocket' object has no attribute 'create_object'


AttributeError: 'PointMassRocket' object has no attribute 'create_object'

### Analyze Results

In [ ]:
# Display statistical summary
print("\n" + "="*60)
print("MONTE CARLO RESULTS SUMMARY")
print("="*60)

for param in ['apogee', 'apogee_time', 'max_speed', 'x_impact', 'y_impact', 'impact_velocity']:
    if param in mc.processed_results:
        mean_val = mc.processed_results[param][0]
        std_val = mc.processed_results[param][1]
        print(f"{param:20s}: {mean_val:10.2f} ± {std_val:8.2f}")

### Visualize Dispersion

Let's create scatter plots to visualize the trajectory dispersion.

In [ ]:
# Extract results
apogees = mc.results['apogee']
x_impacts = mc.results['x_impact']
y_impacts = mc.results['y_impact']
max_speeds = mc.results['max_speed']

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Apogee histogram
axes[0, 0].hist(apogees, bins=20, color='skyblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(nominal_flight.apogee, color='red', linestyle='--', linewidth=2, label='Nominal')
axes[0, 0].axvline(np.mean(apogees), color='green', linestyle='-', linewidth=2, label='Mean')
axes[0, 0].set_xlabel('Apogee Altitude (m)', fontsize=12)
axes[0, 0].set_ylabel('Frequency', fontsize=12)
axes[0, 0].set_title('Apogee Distribution', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Impact scatter plot
axes[0, 1].scatter(x_impacts, y_impacts, alpha=0.6, s=50, c='blue')
axes[0, 1].scatter([nominal_flight.x_impact], [nominal_flight.y_impact], 
                   color='red', s=200, marker='*', label='Nominal', zorder=5)
axes[0, 1].scatter([np.mean(x_impacts)], [np.mean(y_impacts)], 
                   color='green', s=200, marker='X', label='Mean', zorder=5)
axes[0, 1].set_xlabel('Impact X (m East)', fontsize=12)
axes[0, 1].set_ylabel('Impact Y (m North)', fontsize=12)
axes[0, 1].set_title('Landing Dispersion', fontsize=14, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].axis('equal')

# Max speed histogram
axes[1, 0].hist(max_speeds, bins=20, color='lightcoral', edgecolor='black', alpha=0.7)
axes[1, 0].axvline(nominal_flight.max_speed, color='red', linestyle='--', linewidth=2, label='Nominal')
axes[1, 0].axvline(np.mean(max_speeds), color='green', linestyle='-', linewidth=2, label='Mean')
axes[1, 0].set_xlabel('Maximum Speed (m/s)', fontsize=12)
axes[1, 0].set_ylabel('Frequency', fontsize=12)
axes[1, 0].set_title('Max Speed Distribution', fontsize=14, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Range vs apogee correlation
ranges = np.sqrt(np.array(x_impacts)**2 + np.array(y_impacts)**2)
axes[1, 1].scatter(apogees, ranges, alpha=0.6, s=50, c='purple')
axes[1, 1].set_xlabel('Apogee (m)', fontsize=12)
axes[1, 1].set_ylabel('Range from Launch (m)', fontsize=12)
axes[1, 1].set_title('Apogee vs Range Correlation', fontsize=14, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('mc_3dof_flight_variation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nVisualization saved as 'mc_3dof_flight_variation.png'")

## Approach 2: Attempting Rocket Parameter Variation (With Issues)

This section demonstrates the **current limitations** when trying to vary rocket parameters in 3-DOF Monte Carlo.

### The Issue

When we try to use `StochasticRocket` with a `PointMassRocket`, we encounter:

```python
AttributeError: 'PointMassRocket' object has no attribute 'create_object'
```

This is because:
1. Monte Carlo expects stochastic objects that have a `create_object()` method
2. `StochasticRocket` expects a regular `Rocket`, not a `PointMassRocket`
3. There is no `StochasticPointMassRocket` class implemented

### Workaround: Use Regular Rocket with StochasticRocket

We can work around this by using a regular `Rocket` (6-DOF) with `StochasticRocket`, 
then forcing 3-DOF mode in the flight. However, this is not ideal and defeats the purpose.

In [ ]:
# This cell demonstrates the limitation
# Uncomment to see the error:

# from rocketpy.stochastic import StochasticRocket
#
# # This will fail:
# stochastic_rocket_attempt = StochasticRocket(
#     rocket=rocket,  # PointMassRocket
#     mass=(5.0, 0.5, 'normal'),
# )
#
# mc_fail = MonteCarlo(
#     filename="mc_3dof_fail",
#     environment=stochastic_env,
#     rocket=stochastic_rocket_attempt,  # This causes issues
#     flight=stochastic_flight,
# )
#
# # This will raise: AttributeError: 'PointMassRocket' object has no attribute 'create_object'
# mc_fail.simulate(number_of_simulations=5)

print("This cell is commented out to prevent errors.")
print("Uncomment to see the AttributeError when using StochasticRocket with PointMassRocket.")

## Recommendations for 3-DOF Monte Carlo

### What Works Well ✅

1. **Flight parameter variations**: Inclination, heading, rail length
2. **Environment variations**: Using `StochasticEnvironment`
3. **Fast simulations**: 3-DOF enables 100+ simulations quickly
4. **Landing dispersion analysis**: Great for impact zone studies

### Current Limitations ⚠️

1. **No rocket parameter randomization**: Can't vary mass, drag, etc. for PointMassRocket
2. **No motor parameter randomization**: Can't vary thrust, burn time, etc. for PointMassMotor
3. **No StochasticPointMassRocket**: Would need to be implemented

### Recommended Use Cases

3-DOF Monte Carlo is ideal for:
- **Launch angle/heading uncertainty studies**
- **Wind sensitivity analysis** (with StochasticEnvironment)
- **Landing zone prediction**
- **Quick trajectory dispersion studies**

For parameter variations in rocket/motor properties, use 6-DOF Monte Carlo with full `Rocket` and `Motor` classes.

### Future Improvements

To fully support 3-DOF Monte Carlo, the following could be implemented:
1. `StochasticPointMassRocket` class
2. `StochasticPointMassMotor` class
3. Integration with the existing Monte Carlo framework

## Cleanup

In [ ]:
# Clean up generated files
import os

files_to_remove = [
    "mc_3dof_flight_only.inputs.txt",
    "mc_3dof_flight_only.outputs.txt",
    "mc_3dof_flight_only.errors.txt",
]

for f in files_to_remove:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed: {f}")

print("\nCleanup complete!")

## Conclusion

This notebook demonstrated:

✅ **Working approach**: Monte Carlo with 3-DOF using flight parameter variations  
⚠️ **Current limitation**: Cannot vary rocket/motor parameters with PointMassRocket  
📊 **Statistical analysis**: Mean, std deviation, and distribution visualization  
🎯 **Landing dispersion**: Impact zone analysis and scatter plots  

### Key Takeaway

While 3-DOF Monte Carlo has some limitations regarding parameter randomization, 
it is **highly effective** for:
- Launch uncertainty analysis
- Fast trajectory dispersion studies  
- Environmental sensitivity studies

For comprehensive uncertainty quantification including rocket and motor parameters, 
use 6-DOF simulations with the full `Rocket` and `Motor` classes.